In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
from kaggle.api.kaggle_api_extended import KaggleApi
import zipfile

# Authenticate
api = KaggleApi()
api.authenticate()

# Download dataset
dataset = "maharshipandya/-spotify-tracks-dataset"
api.dataset_download_files(dataset, path="data", unzip=True)

# Load file
file_path = "data/Reviews.csv"


songs = pd.read_csv("data/dataset.csv")
songs = songs.drop(columns=["Unnamed: 0", "Id"], errors="ignore")
songs = songs.fillna(0)
users = users.fillna(0)



# Load user-song interaction file
users = pd.read_csv("user_artists.dat", sep="\t")

# # Show data
# print(data.head())
# print(data.columns)

Dataset URL: https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset


NameError: name 'users' is not defined

In [60]:
# SEARCH TRACKING SYSTEM

search_count = 0

# store how many times each song is searched
song_search_counter = {}


# 2. CONTENT MODEL SETUP

scaler = StandardScaler()
song_model = NearestNeighbors(metric='cosine', algorithm='brute')
scaled_features = None

def rebuild_content_model():
    """Fits the scaler and KNN model on the current songs DataFrame."""
    global scaled_features, song_model, scaler
    features = songs[
        ["danceability", "energy", "acousticness",
         "instrumentalness", "liveness", "valence", "tempo"]
    ]
    scaled_features = scaler.fit_transform(features)
    song_model.fit(scaled_features)

# Initial build
rebuild_content_model()




In [61]:
#  USER MODEL
def rebuild_user_matrix():
    return users.pivot_table(
        index="userID",
        columns="artistID",
        values="weight",
        fill_value=0
    )

user_matrix = rebuild_user_matrix()
user_similarity = cosine_similarity(user_matrix)


# PRETTY PRINT

def clean(df, title):

    print("\n" + "=" * 70)
    print(f" {title}")
    print("=" * 70)

    if isinstance(df, str):
        print(df)
        return

    if df is None or len(df) == 0:
        print("No data found.")
        return

    print(df.to_string(index=False))
    print("=" * 70)


In [62]:
#  SONG RECOMMENDATION

def recommend_song(song_name, top_n=10):

    if song_name not in songs["track_name"].values:
        return None

    idx = songs[songs["track_name"] == song_name].index[0]

    # Ensure we don't request more neighbors than available songs
    actual_n = min(top_n + 1, len(songs))

    distances, indices = song_model.kneighbors(
        [scaled_features[idx]],
        n_neighbors=actual_n
    )

    result = songs.iloc[indices.flatten()[1:]][
        ["track_name", "artists", "track_genre"]
    ].copy()

    # Map the search count from our global dictionary to each recommended song
    result["search_count"] = result["track_name"].map(song_search_counter).fillna(0).astype(int)
    result["rank"] = range(1, len(result) + 1)
    result["searched_song"] = song_name

    # Reorder columns to display search count visibly
    result = result[["rank", "track_name", "artists", "track_genre", "search_count", "searched_song"]]

    return result





In [63]:
#  UPDATE USER DATA (SMART LEARNING)

def update_user_data(user_id, song_name):

    global users

    song_row = songs[songs["track_name"] == song_name]

    # NEW SONG CASE
    if song_row.empty:
        users = pd.concat([users, pd.DataFrame({
            "userID": [user_id],
            "artistID": ["UNKNOWN"],
            "weight": [1]
        })], ignore_index=True)
        return

    artist = song_row["artists"].values[0]
    mask = (users["userID"] == user_id) & (users["artistID"] == artist)

    if users[mask].empty:
        users = pd.concat([users, pd.DataFrame({
            "userID": [user_id],
            "artistID": [artist],
            "weight": [1]
        })], ignore_index=True)
    else:
        #  increase based on search frequency
        users.loc[mask, "weight"] += 1




In [64]:
#  UPDATE SEARCH COUNTER SYSTEM

def track_song_search(song_name):

    global song_search_counter

    if song_name in song_search_counter:
        song_search_counter[song_name] += 1
    else:
        song_search_counter[song_name] = 1

    return song_search_counter[song_name]

In [65]:
#  TOP RECOMMENDATIONS (DYNAMIC)

def top_recommendations(user_id, top_n=10):

    global user_matrix, user_similarity

    user_matrix = rebuild_user_matrix()
    
    # Handle edge case where matrix might be empty or user missing
    if user_id not in user_matrix.index:
        return "User not found."
        
    user_similarity = cosine_similarity(user_matrix)

    # 1. Sort global dictionary to find the most heavily searched tracks
    popular_songs = sorted(song_search_counter.items(),
                           key=lambda x: x[1],
                           reverse=True)

    popular_df = pd.DataFrame(popular_songs, columns=["track_name", "search_count"])

    # 2. Fetch data details for those popular tracks
    if not popular_df.empty:
        popular_merge = songs[songs["track_name"].isin(popular_df["track_name"])][
            ["track_name", "artists", "track_genre"]
        ].copy()
        
        # Attach precise count so it displays nicely
        popular_merge["search_count"] = popular_merge["track_name"].map(song_search_counter).fillna(0).astype(int)
        # Prioritize sorting by most searched tracks descending
        popular_merge = popular_merge.sort_values(by="search_count", ascending=False)
    else:
        popular_merge = pd.DataFrame(columns=["track_name", "artists", "track_genre", "search_count"])

    # 3. Create a fallback base of random tracks
    base_rec = songs.sample(min(top_n, len(songs)))[
        ["track_name", "artists", "track_genre"]
    ].copy()
    base_rec["search_count"] = base_rec["track_name"].map(song_search_counter).fillna(0).astype(int)

    # 4. Inject highly searched songs at the absolute top of the feed
    final = pd.concat([popular_merge, base_rec]).drop_duplicates(subset=["track_name"])

    return final.head(top_n)

In [ ]:
#  MAIN SYSTEM

def run_system(user_id):

    global search_count, songs

    clean(top_recommendations(user_id), "INITIAL RECOMMENDATIONS")

    while True:

        song = input("\n Enter song name (or exit): ").strip()

        if not song:
            continue

        if song.lower() == "exit":
            break

        search_count += 1

        print("\n" + "=" * 70)
        print(f" TOTAL SYSTEM SEARCH COUNT: {search_count}")
        print("=" * 70)

        #  track song searches
        count = track_song_search(song)
        print(f" This song has been searched {count} times total.")

        # Check if the song exists in our library
        if song not in songs["track_name"].values:
            # Add new song row with dummy feature values
            new_song_row = pd.DataFrame([{
                "track_name": song, 
                "artists": "UNKNOWN", 
                "track_genre": "UNKNOWN",
                "danceability": 0, "energy": 0, "acousticness": 0, 
                "instrumentalness": 0, "liveness": 0, "valence": 0, "tempo": 0
            }])
            songs = pd.concat([songs, new_song_row], ignore_index=True)
            
            #  CRITICAL: Re-fit structural vectorizer models with the added data point
            rebuild_content_model()
            
            print("\n New song detected → added to the system database!")
        
        # Pull recommendation index using content model
        result = recommend_song(song)
        

        if result is not None:
            clean(result, "TOP SIMILAR SONGS (WITH SEARCH COUNTS)")

        update_user_data(user_id, song)

        # Print out the revised user dashboard feed
        clean(top_recommendations(user_id), "UPDATED RECOMMENDATIONS (POPULAR TRACKS BUMPED UP)")


#  RUN SYSTEM

if __name__ == "__main__":
    run_system(2)


 INITIAL RECOMMENDATIONS
                        track_name                                  artists track_genre search_count
        The Flame - Single Version                              Cheap Trick   power-pop            0
               Child of the Desert                            Circa Survive    hardcore            0
                  Christmas Lights                                 Coldplay         pop            0
                 Trillando la fina                               Almafuerte heavy-metal            0
      Canção Ao Cordeiro - Ao Vivo Israel Salazar;Gabriel Guedes de Almeida      brazil            0
                   Feel Invincible                                  Skillet        rock            0
                        Porno Shop                               Teen Idols   power-pop            0
                         Territory                                Sepultura      groove            0
Sun In Your Eyes - Sunny Lax Remix                 Above & Beyond


 Enter song name (or exit):  Sleep My Little Boy



 TOTAL SYSTEM SEARCH COUNT: 1
 This song has been searched 1 times total.

 TOP SIMILAR SONGS (WITH SEARCH COUNTS)
 rank                                       track_name                                                               artists       track_genre  search_count       searched_song
    1                                             千軍万馬                                                    Yasuharu Takanashi             anime             0 Sleep My Little Boy
    2                                       Echosphere                                                             Alphaxone           iranian             0 Sleep My Little Boy
    3        To the Stars - From "Ad Astra" Soundtrack                                                           Max Richter           ambient             0 Sleep My Little Boy
    4                           Pink Noise Tranquility                                                            Pink Noise             sleep             0 Sleep My Little Boy


 Enter song name (or exit):  To Begin Again



 TOTAL SYSTEM SEARCH COUNT: 2
 This song has been searched 1 times total.

 TOP SIMILAR SONGS (WITH SEARCH COUNTS)
 rank               track_name                artists track_genre  search_count  searched_song
    1 Feel Again (Feat. Au/Ra)             Kina;Au/Ra         sad             0 To Begin Again
    2   How Great Is Your Love           Phil Wickham world-music             0 To Begin Again
    3              Infatuation                 SOPHIE        club             0 To Begin Again
    4          Tempo de Vencer                 Jamily      gospel             0 To Begin Again
    5          Tempo de Vencer                 Jamily      brazil             0 To Begin Again
    6    Hymn Of The Big Wheel         Massive Attack    trip-hop             0 To Begin Again
    7        You Saved My Soul  Bryan & Katie Torwalt world-music             0 To Begin Again
    8         Linda Mi Cholita           William Luna      guitar             0 To Begin Again
    9                Bote o P

...


 TOTAL SYSTEM SEARCH COUNT: 3
 This song has been searched 1 times total.

 TOP SIMILAR SONGS (WITH SEARCH COUNTS)
 rank                                                     track_name                                                            artists track_genre  search_count    searched_song
    1                                  Musica para Relajacion Guiada                                                      Reiki Armonía world-music             0 Water Into Light
    2                                                 Rios y Arroyos                                                        Agua Mantra world-music             0 Water Into Light
    3                                             Realms of Splendor                                                               2002     new-age             0 Water Into Light
    4                                                    Butterflies                                                       Porya Hatami     iranian             0 Water 


 Enter song name (or exit):  To Begin Again



 TOTAL SYSTEM SEARCH COUNT: 4
 This song has been searched 2 times total.

 TOP SIMILAR SONGS (WITH SEARCH COUNTS)
 rank               track_name                artists track_genre  search_count  searched_song
    1 Feel Again (Feat. Au/Ra)             Kina;Au/Ra         sad             0 To Begin Again
    2   How Great Is Your Love           Phil Wickham world-music             0 To Begin Again
    3              Infatuation                 SOPHIE        club             0 To Begin Again
    4          Tempo de Vencer                 Jamily      gospel             0 To Begin Again
    5          Tempo de Vencer                 Jamily      brazil             0 To Begin Again
    6    Hymn Of The Big Wheel         Massive Attack    trip-hop             0 To Begin Again
    7        You Saved My Soul  Bryan & Katie Torwalt world-music             0 To Begin Again
    8         Linda Mi Cholita           William Luna      guitar             0 To Begin Again
    9                Bote o P


 Enter song name (or exit):  Water Into Light



 TOTAL SYSTEM SEARCH COUNT: 5
 This song has been searched 2 times total.

 TOP SIMILAR SONGS (WITH SEARCH COUNTS)
 rank                                                     track_name                                                            artists track_genre  search_count    searched_song
    1                                  Musica para Relajacion Guiada                                                      Reiki Armonía world-music             0 Water Into Light
    2                                                 Rios y Arroyos                                                        Agua Mantra world-music             0 Water Into Light
    3                                             Realms of Splendor                                                               2002     new-age             0 Water Into Light
    4                                                    Butterflies                                                       Porya Hatami     iranian             0 Water 

In [67]:
songs

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.7150,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.2670,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.1200,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.1430,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.1670,119.949,4,acoustic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113995,2C3TZjDRiAzdyViavDJ217,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Sleep My Little Boy,21,384999,False,0.172,0.2350,5,-16.393,1,0.0422,0.6400,0.928000,0.0863,0.0339,125.995,5,world-music
113996,1hIz5L4IB9hN3WRYPOCGPw,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Water Into Light,22,385000,False,0.174,0.1170,0,-18.318,0,0.0401,0.9940,0.976000,0.1050,0.0350,85.239,4,world-music
113997,6x8ZfSoqDjuNa5SVP5QjvX,Cesária Evora,Best Of,Miss Perfumado,22,271466,False,0.629,0.3290,0,-10.895,0,0.0420,0.8670,0.000000,0.0839,0.7430,132.378,4,world-music
113998,2e6sXL2bYv4bSz6VTdnfLs,Michael W. Smith,Change Your World,Friends,41,283893,False,0.587,0.5060,7,-10.889,1,0.0297,0.3810,0.000000,0.2700,0.4130,135.960,4,world-music
